In [1]:

from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

DATA_DIR = Path("../data/raw/NEU-DET")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

def get_images(split):
    root = DATA_DIR / split / "images"
    return [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

def audit_split(split):
    image_paths = get_images(split)
    annotation_paths = list(
        (DATA_DIR / split / "annotations").rglob("*.xml")
    )

    # Index by stem. Preserve duplicates rather than silently overwriting.
    image_index = {}
    for image_path in image_paths:
        image_index.setdefault(
            image_path.stem.lower(), []
        ).append(image_path)

    report = []
    unmatched = []
    multi_class = []
    invalid_boxes = []

    for xml_path in annotation_paths:
        root = ET.parse(xml_path).getroot()
        filename = root.findtext("filename", "")
        stem = Path(filename).stem.lower() if filename else ""

        candidates = image_index.get(stem, [])

        if len(candidates) != 1:
            unmatched.append({
                "xml": xml_path.name,
                "filename": filename,
                "candidate_count": len(candidates),
            })
            continue

        image_path = candidates[0]
        labels = []

        size = root.find("size")
        width = int(size.findtext("width"))
        height = int(size.findtext("height"))

        for obj in root.findall("object"):
            label = obj.findtext("name", "").strip()
            box = obj.find("bndbox")

            xmin = int(box.findtext("xmin"))
            ymin = int(box.findtext("ymin"))
            xmax = int(box.findtext("xmax"))
            ymax = int(box.findtext("ymax"))

            labels.append(label)

            if not (
                0 <= xmin < xmax <= width
                and 0 <= ymin < ymax <= height
            ):
                invalid_boxes.append(xml_path.name)

        if len(set(labels)) > 1:
            multi_class.append({
                "xml": xml_path.name,
                "image": str(image_path),
                "labels": labels,
            })

        report.append({
            "xml": xml_path.name,
            "image": str(image_path),
            "folder_class": image_path.parent.name,
            "filename_prefix": xml_path.stem.rsplit("_", 1)[0],
            "labels": labels,
        })

    print(f"\n--- {split.upper()} ---")
    print("Images:", len(image_paths))
    print("XML files:", len(annotation_paths))
    print("Unique image matches:", len(report))
    print("Unmatched or ambiguous:", len(unmatched))
    print("Multi-class annotations:", len(multi_class))
    print("Invalid boxes:", len(invalid_boxes))

    print("\nUnmatched examples:")
    for row in unmatched[:10]:
        print(row)

    print("\nMulti-class examples:")
    for row in multi_class[:10]:
        print(row)

    print("\nFolder class versus annotation labels:")
    disagreements = [
        row for row in report
        if row["folder_class"] not in row["labels"]
    ]
    print("Disagreements:", len(disagreements))
    for row in disagreements[:10]:
        print(row)

    return report, unmatched, multi_class, disagreements


train_report, train_unmatched, train_multi, train_disagreements = (
    audit_split("train")
)

val_report, val_unmatched, val_multi, val_disagreements = (
    audit_split("validation")
)



--- TRAIN ---
Images: 1440
XML files: 1439
Unique image matches: 1439
Unmatched or ambiguous: 0
Multi-class annotations: 100
Invalid boxes: 0

Unmatched examples:

Multi-class examples:
{'xml': 'crazing_104.xml', 'image': '..\\data\\raw\\NEU-DET\\train\\images\\crazing\\crazing_104.jpg', 'labels': ['crazing', 'crazing', 'crazing', 'patches']}
{'xml': 'crazing_105.xml', 'image': '..\\data\\raw\\NEU-DET\\train\\images\\crazing\\crazing_105.jpg', 'labels': ['crazing', 'crazing', 'crazing', 'inclusion']}
{'xml': 'crazing_112.xml', 'image': '..\\data\\raw\\NEU-DET\\train\\images\\crazing\\crazing_112.jpg', 'labels': ['crazing', 'crazing', 'patches']}
{'xml': 'crazing_152.xml', 'image': '..\\data\\raw\\NEU-DET\\train\\images\\crazing\\crazing_152.jpg', 'labels': ['crazing', 'crazing', 'patches']}
{'xml': 'crazing_20.xml', 'image': '..\\data\\raw\\NEU-DET\\train\\images\\crazing\\crazing_20.jpg', 'labels': ['crazing', 'crazing', 'patches']}
{'xml': 'patches_155.xml', 'image': '..\\data\\raw\